# STREME / TOMTOM / FIMO pipeline

This notebook configures, runs and summarizes the motif pipeline. The reusable
implementation is in `streme_pipeline.py`; the same code is used by the Slurm
array jobs.


## Setup


In [55]:
from pathlib import Path
import pandas as pd

from streme_pipeline import (
    PipelineConfig,
    PipelineParameters,
    STAGES,
    build_status,
    combine_result_tables,
    discover_fastas,
    export_fimo_for_binding_bench,
    run_all_stages,
    run_pipeline_for_fasta,
    save_analysis_tables,
    save_summaries,
)


In [56]:
config = PipelineConfig.default(
    project=Path("/s/project/ml4rg_students/2026/project15")
)
config.validate()

fastas = discover_fastas(config.fasta_dir)
parameters = PipelineParameters(
    streme_time=1800,
    minw=6,
    maxw=20,
    nmotifs=10,
    fimo_thresh="1e-4",
    fimo_max_stored_scores=100_000,
    fimo_skip_matched_sequence=False,
)

print(f"Found {len(fastas)} FASTA files")
print("FASTA directory:", config.fasta_dir)
print("Result directory:", config.result_dir)
print("JASPAR database:", config.jaspar_fungi)


Found 1401 FASTA files
FASTA directory: /s/project/ml4rg_students/2026/project15/working/sequence_datasets_fastas
Result directory: /s/project/ml4rg_students/2026/project15/working/streme_results
JASPAR database: /s/project/ml4rg_students/2026/project15/working/jaspar/JASPAR2026_CORE_fungi_non-redundant_pfms_meme.txt


The pipeline writes a `.pipeline_done.json` next to each successful
result. New results are only reused when their command and input signatures
match. Existing results from the old notebook have no manifest and are accepted
as a legacy cache by default.


## Status


In [57]:
!squeue -j 19470729

slurm_load_jobs error: Invalid job id specified


In [58]:
!sacct -X -j 19470729 --format=JobID,JobName,State,ExitCode,Elapsed,MaxRSS

JobID           JobName      State ExitCode    Elapsed     MaxRSS 
------------ ---------- ---------- -------- ---------- ---------- 
19470729_0   streme-st+  COMPLETED      0:0   00:00:03            
19470729_1   streme-st+  COMPLETED      0:0   00:00:03            
19470729_2   streme-st+  COMPLETED      0:0   00:00:03            
19470729_3   streme-st+  COMPLETED      0:0   00:00:03            
19470729_4   streme-st+  COMPLETED      0:0   00:00:03            
19470729_5   streme-st+  COMPLETED      0:0   00:00:03            
19470729_6   streme-st+  COMPLETED      0:0   00:00:03            
19470729_7   streme-st+  COMPLETED      0:0   00:00:03            
19470729_8   streme-st+  COMPLETED      0:0   00:00:03            
19470729_9   streme-st+  COMPLETED      0:0   00:00:02            
19470729_10  streme-st+  COMPLETED      0:0   00:00:02            
19470729_11  streme-st+  COMPLETED      0:0   00:00:02            
19470729_12  streme-st+  COMPLETED      0:0   00:00:03        

In [59]:
status = build_status(config, fastas)
display(status.head())
display(status[list(STAGES)].sum().rename("completed"))

incomplete = status.loc[~status[list(STAGES)].all(axis=1)]
print(f"Incomplete datasets: {len(incomplete)}")
display(incomplete.head(20))


,name,fasta,streme,tomtom,fimo_jaspar,fimo_streme
0,_candida_arabinofermentans_nrrl_yb_2248_gca_00...,/s/project/ml4rg_students/2026/project15/worki...,True,True,True,True
1,_candida_auris_gca_001189475_sequence_mapper,/s/project/ml4rg_students/2026/project15/worki...,True,True,True,True
2,_candida_auris_gca_002775015_sequence_mapper,/s/project/ml4rg_students/2026/project15/worki...,True,True,True,True
3,_candida_auris_gca_003013715_sequence_mapper,/s/project/ml4rg_students/2026/project15/worki...,True,True,True,True
4,_candida_auris_gca_003014415_sequence_mapper,/s/project/ml4rg_students/2026/project15/worki...,True,True,True,True


streme         497
fimo_jaspar     74
tomtom          62
fimo_streme     24
Name: completed, dtype: int64

Incomplete datasets: 1377


,name,fasta,streme,tomtom,fimo_jaspar,fimo_streme
23,absidia_repens_gca_002105175_sequence_mapper,/s/project/ml4rg_students/2026/project15/worki...,True,True,True,False
24,acaromyces_ingoldii_gca_003144295_sequence_mapper,/s/project/ml4rg_students/2026/project15/worki...,True,True,True,False
25,acidomyces_richmondensis_bfw_gca_001592465_seq...,/s/project/ml4rg_students/2026/project15/worki...,True,True,True,False
26,acidomyces_sp_richmondensis_gca_001572075_sequ...,/s/project/ml4rg_students/2026/project15/worki...,True,True,True,False
27,acremonium_chrysogenum_atcc_11550_gca_00076926...,/s/project/ml4rg_students/2026/project15/worki...,True,True,True,False
28,agaricus_bisporus_var_burnettii_jb137_s8_gca_0...,/s/project/ml4rg_students/2026/project15/worki...,True,True,True,False
29,agrocybe_aegerita_gca_902728275_sequence_mapper,/s/project/ml4rg_students/2026/project15/worki...,True,True,True,False
31,akanthomyces_lecanii_rcef_1005_gca_001636795_s...,/s/project/ml4rg_students/2026/project15/worki...,True,True,True,False
32,allomyces_macrogynus_atcc_38327_gca_000151295_...,/s/project/ml4rg_students/2026/project15/worki...,True,True,True,False
33,alternaria_alternata_gca_001642055_sequence_ma...,/s/project/ml4rg_students/2026/project15/worki...,True,True,True,False


In [60]:
from pathlib import Path
import pyarrow.parquet as pq

project_dir = Path("/s/project/ml4rg_students/2026/project15")

parquet_file = ("/s/project/multispecies/fungi_code/tf_sae/binding_bench_datasets/val/dna/_saccharomyces_cerevisiae/DNA_rossi_chipexo_sites.parquet"
)

output_dir = project_dir / "working" / "sequence_datasets_fastas"
output_dir.mkdir(parents=True, exist_ok=True)

out_path = output_dir / ("DNA_rossi_chipexo_sites.fasta")

table = pq.read_table(
    parquet_file,
    columns=["chrom", "start", "end", "strand", "seq", "gene_id"],
)
df = table.to_pandas()

with open(out_path, "w") as out:
    for _, row in df.iterrows():
        seq = str(row["seq"]).upper()
        seq = "".join(c if c in "ACGTN" else "N" for c in seq)

        if len(seq) == 0:
            continue

        header = (
            f"{row['gene_id']}"
            f"|species_file={parquet_file.stem}"
            f"|chr={row['chrom']}"
            f"|start={row['start']}"
            f"|end={row['end']}"
            f"|strand={row['strand']}"
        )

        out.write(f">{header}\n")
        for j in range(0, len(seq), 80):
            out.write(seq[j:j+80] + "\n")

print("Wrote:", out_path)

ArrowInvalid: No match for FieldRef.Name(seq) in name: large_string
split: large_string
tf_class: large_string
chrom: large_string
start: int64
end: int64
score: large_string
strand: large_string
binding_evidence: bool
motif_evidence: bool
motif_source: large_string
binding_score: int64
match_qval: double
match_start: int64
match_end: int64
site_id: large_string
d_peak: int64
nearest_peak_id: large_string
tf_class_right: large_string
tf_family: large_string
__fragment_index: int32
__batch_index: int32
__last_in_fragment: bool
__filename: string

## Test one FASTA


In [61]:
from pathlib import Path

test_fasta = Path(
    "/s/project/ml4rg_students/2026/project15/working/"
    "sequence_datasets_fastas/_saccharomyces_cerevisiae_sequence_mapper.fasta"
)

test_result = run_pipeline_for_fasta(
    config,
    test_fasta,
    parameters=parameters,
    force=False,
    accept_legacy=True,
)

test_result


[_saccharomyces_cerevisiae_sequence_mapper] streme: existing legacy result accepted
[_saccharomyces_cerevisiae_sequence_mapper] fimo_jaspar: cached
[_saccharomyces_cerevisiae_sequence_mapper] tomtom: cached
[_saccharomyces_cerevisiae_sequence_mapper] fimo_streme: cached


{'streme': 'legacy',
 'fimo_jaspar': 'cached',
 'tomtom': 'cached',
 'fimo_streme': 'cached'}

In [62]:
analysis_paths = save_analysis_tables(
    config,
    fastas,
    tomtom_q_value=0.05,
)
analysis_paths


KeyboardInterrupt: 

In [63]:
prediction_path = export_fimo_for_binding_bench(
    config,
    test_fasta,
    result_key="fimo_streme_tsv",
)

prediction_path

PosixPath('/s/project/ml4rg_students/2026/project15/working/streme_results/binding_bench/_saccharomyces_cerevisiae_sequence_mapper_fimo_streme_tsv.tsv')

In [64]:
!find /data/nasif12/home_if12/s_kmill -name pyproject.toml -path "*/binding_bench/*" 2>/dev/null

/data/nasif12/home_if12/s_kmill/binding_bench/pyproject.toml


In [65]:
!pip install -e /data/nasif12/home_if12/s_kmill/binding_bench

Obtaining file:///data/nasif12/home_if12/s_kmill/binding_bench
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Installing backend dependencies ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for binding_bench (pyproject.toml) ... done
  Created wheel for binding_bench: filename=binding_bench-0.0.1-py2.py3-none-any.whl size=1309 sha256=e1e8dc4fec1072b94c65b942c87a291b5c382eb4bde9ef4298b3f9e559ab772e
  Stored in directory: /scratch/tmp/s_kmill/pip-ephem-wheel-cache-vuq4e2_q/wheels/a8/b4/e0/4b7cf52c0bdccd67020c28e7d5ca9092188f95c689f63e4d8e
Successfully built binding_bench
  Attempting uninstall: binding_bench
    Found existing installation: binding_bench 0.0.1
    Uninstalling binding_bench-0.0.1:
      Successfully uninstalled binding_bench-0.0.1


In [66]:
!python -m binding_bench discrete \
  --input /s/project/ml4rg_students/2026/project15/working/streme_results/binding_bench/_saccharomyces_cerevisiae_sequence_mapper_fimo_streme_tsv.tsv \
  --dataset DNA_rossi_chipexo \
  --output_dir /s/project/ml4rg_students/2026/project15/working/streme_results/binding_bench/streme_run \
  --metric jaccard precision_lb recall_lb \
  --best_assignment \
  --display_name STREME-FIMO \
  --overwrite

In [67]:
!ls -lh /s/project/ml4rg_students/2026/project15/working/streme_results/binding_bench/_saccharomyces_cerevisiae_sequence_mapper_fimo_streme_tsv.tsv

-rw-rw----+ 1 s_kmill root 4.8M Jun 18 13:56 /s/project/ml4rg_students/2026/project15/working/streme_results/binding_bench/_saccharomyces_cerevisiae_sequence_mapper_fimo_streme_tsv.tsv


In [68]:
!head /s/project/ml4rg_students/2026/project15/working/streme_results/binding_bench/_saccharomyces_cerevisiae_sequence_mapper_fimo_streme_tsv.tsv

chrom	start	end	feature_idx	score	strand
XV	444507	444508	1-GAAAAAAAAAAAAAAAAAA	10.752026733638193	-
X	136919	136920	1-GAAAAAAAAAAAAAAAAAA	10.752026733638193	+
XIII	851722	851723	1-GAAAAAAAAAAAAAAAAAA	10.752026733638193	+
IX	241041	241042	1-GAAAAAAAAAAAAAAAAAA	10.752026733638193	+
VIII	187038	187039	1-GAAAAAAAAAAAAAAAAAA	10.752026733638193	+
I	70915	70916	1-GAAAAAAAAAAAAAAAAAA	10.752026733638193	+
XVI	840644	840645	1-GAAAAAAAAAAAAAAAAAA	10.752026733638193	-
II	145907	145908	1-GAAAAAAAAAAAAAAAAAA	10.752026733638193	+
VII	733121	733122	1-GAAAAAAAAAAAAAAAAAA	10.752026733638193	-


In [69]:
!find /s/project/ml4rg_students/2026/project15/working/streme_results/binding_bench/streme_run -type f

find: ‘/s/project/ml4rg_students/2026/project15/working/streme_results/binding_bench/streme_run’: No such file or directory


In [70]:
!python -m binding_bench discrete --help

In [71]:
!ls -lh /s/project/ml4rg_students/2026/project15/working/streme_results/binding_bench/

total 4.8M
-rw-rw----+ 1 s_kmill root 4.8M Jun 18 13:56 _saccharomyces_cerevisiae_sequence_mapper_fimo_streme_tsv.tsv


In [72]:
!mkdir -p /s/project/ml4rg_students/2026/project15/working/streme_results/binding_bench/streme_run

!python -m binding_bench discrete \
  --input /s/project/ml4rg_students/2026/project15/working/streme_results/binding_bench/_saccharomyces_cerevisiae_sequence_mapper_fimo_streme_tsv.tsv \
  --dataset DNA_rossi_chipexo \
  --output_dir /s/project/ml4rg_students/2026/project15/working/streme_results/binding_bench/streme_run \
  --metric jaccard precision_lb recall_lb \
  --best_assignment \
  --display_name STREME-FIMO \
  --overwrite

In [73]:
!find /s/project/ml4rg_students/2026/project15/working/streme_results/binding_bench/streme_run -type f